# 05. 6-Way Multilingual Translation Instruction Generator
Demonstrates `InstructionTaskGenerator` and `LexicalTaskGenerator` on a sample of the corpus (full-scale generation is run via `scripts/generate_tasks.sh`, ~187K tasks for the full training split).

In [ ]:
# ============================================================
# PATH & REPO AUTO-SYNC BOOSTER — Guarantees latest project code
# ============================================================
import os, sys, site, urllib.request, zipfile

# Purge cached 'src' modules from memory so updated files take effect immediately
for mod in list(sys.modules.keys()):
    if mod.startswith('src'):
        del sys.modules[mod]

user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.insert(0, user_site)

try:
    cwd = os.getcwd()
except FileNotFoundError:
    cwd = os.path.expanduser('~')
    os.chdir(cwd)

home       = os.path.expanduser('~')
proj_dir   = os.path.join(home, 'Ekegusii-LLM-Translation-main')
sync_tag   = os.path.join(proj_dir, 'configs', 'models', 'v2_mistral_earlystop_v5.tag')

# Auto-sync if folder is missing OR outdated (lacks v2_mistral_earlystop_v5.tag)
if not os.path.isfile(sync_tag):
    print('🔄 Outdated or missing repository detected. Auto-syncing latest code from GitHub...')
    zip_path = os.path.join(home, 'repo.zip')
    urllib.request.urlretrieve('https://github.com/aykahsay/Ekegusii-LLM-Translation/archive/refs/heads/main.zip', zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(home)
    os.remove(zip_path)
    print('✅ Repository auto-synced to latest main commit!')

if os.path.isdir(proj_dir):
    os.chdir(proj_dir)
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f'Working Directory : {os.getcwd()}')
print(f'Python Kernel     : {sys.executable}')


In [2]:
# ============================================================
# ABI CHECK -- numpy/pandas binary compatibility.
# Some Jupyter hosts (e.g. Kineses Cloud conda envs) ship a numpy/pandas
# pair whose compiled C-extension ABI doesn't match, raising
# "numpy.dtype size changed, may indicate binary incompatibility" the
# moment pandas -- and therefore anything importing it, like
# src.master_corpus -- is loaded. Detect and fix it BEFORE any pandas
# import below (see notebooks/00_setup_environment.ipynb for the
# original version of this check).
# ============================================================
import subprocess
import sys


def _abi_ok():
    try:
        import numpy  # noqa: F401
        import pandas  # noqa: F401
        return True
    except ValueError as exc:
        if "binary incompatibility" in str(exc):
            return False
        raise


if not _abi_ok():
    print("numpy/pandas ABI mismatch detected -- attempting fix...")
    fix_a = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "numpy>=2.0.0"],
        capture_output=True, text=True,
    )
    if fix_a.returncode != 0:
        print("  numpy upgrade failed (read-only env?) -- downgrading pandas instead...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "pandas==2.2.3"],
            capture_output=True, text=True,
        )
    raise RuntimeError(
        "Fixed numpy/pandas ABI mismatch via pip -- you MUST restart the kernel now "
        "(Kernel -> Restart Kernel) and re-run this notebook from the top. The fix "
        "cannot take effect in the current running process."
    )
else:
    print("numpy/pandas ABI OK.")


numpy/pandas ABI OK.


In [3]:
from src.master_corpus.manager import MasterCorpusManager
from src.task_generation.instruction_generator import InstructionTaskGenerator
from src.task_generation.lexical_tasks import LexicalTaskGenerator

manager = MasterCorpusManager()
sample = manager.load_train_split().sample(200, random_state=42)

INFO | Loaded dataset split [master_train.csv]: 39,421 rows.


## Sentence-level 6-way tasks

In [4]:
sentence_gen = InstructionTaskGenerator(manager)
tasks_df = sentence_gen.generate_tasks_from_dataframe(sample)
print(f'{len(sample)} concepts -> {len(tasks_df)} instruction tasks')
tasks_df['task_type'].value_counts()

INFO | Expanded 200 concepts into 952 instruction tasks.


200 concepts -> 952 instruction tasks


task_type
ENG_to_EKE    188
EKE_to_ENG    188
SWA_to_ENG    150
ENG_to_SWA    150
EKE_to_SWA    138
SWA_to_EKE    138
Name: count, dtype: int64

In [5]:
tasks_df.iloc[0][['task_type', 'prompt', 'response']]

task_type                                           ENG_to_EKE
prompt       Translate the following English text into Ekeg...
response     Ninkorusie nkorusie ase amaboko ’abakori amabe...
Name: 0, dtype: object

## Lexical-corpus tasks

In [6]:
lexical_gen = LexicalTaskGenerator(manager)
lexical_tasks_df = lexical_gen.generate_all_tasks()
print(f'{len(lexical_tasks_df)} lexical tasks generated')
lexical_tasks_df['task_type'].value_counts()

INFO | Loaded Master Lexical Corpus: 268 terms.


INFO | Generated 536 lexical instruction tasks from 268 lexicon entries.


536 lexical tasks generated


task_type
SWA_to_EKE_lexical    268
EKE_to_SWA_lexical    268
Name: count, dtype: int64